# Performance Estimation Results

Set `WORKLOAD` below to select which input/output configuration to view.

Columns: `InstanceType, TP, PP, BatchSize, E2ELatency, Throughput`

In [ ]:
import json
import glob
import os
import pandas as pd

# ─── Select workload ───────────────────────────────────────────────
WORKLOAD = "in763-out232"  # Change this to view different workloads
# ───────────────────────────────────────────────────────────────────

EST_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), WORKLOAD, "results", "data", "estimated")

# List available workloads
available = sorted([
    d for d in os.listdir(os.path.dirname(os.path.abspath("__file__")))
    if d.startswith("in") and "-out" in d
       and os.path.isdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), d))
])
print(f"Available workloads: {available}")
print(f"Selected: {WORKLOAD}")


def load_estimated(est_dir: str = EST_DIR) -> pd.DataFrame:
    """
    Load all estimation JSONs and flatten batch_sweep into a DataFrame.
    """
    files = sorted(glob.glob(os.path.join(est_dir, "est_*.json")))
    rows = []
    for f in files:
        with open(f) as fp:
            d = json.load(fp)
        if not d.get("feasible", False):
            continue
        instance = d["instance_type"]
        tp = d["tp_size"]
        pp = d["pp_size"]
        for entry in d.get("batch_sweep", []):
            rows.append({
                "InstanceType": instance,
                "TP": tp,
                "PP": pp,
                "BatchSize": entry["batch_size"],
                "E2ELatency": round(entry["batch_latency_ms"], 2),
                "Throughput": round(entry["throughput_rps"], 4),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["InstanceType", "TP", "PP", "BatchSize"]).reset_index(drop=True)
    return df


def filter_df(df, instance=None, tp=None, pp=None):
    """Filter by instance type, TP, and/or PP."""
    if instance:
        df = df[df["InstanceType"] == instance]
    if tp:
        df = df[df["TP"] == tp]
    if pp:
        df = df[df["PP"] == pp]
    return df.reset_index(drop=True)


def max_batch_summary(df):
    """One row per (Instance, TP, PP) — max batch only."""
    return (
        df.loc[df.groupby(["InstanceType", "TP", "PP"])["BatchSize"].idxmax()]
        .sort_values(["InstanceType", "TP", "PP"])
        .reset_index(drop=True)
    )

In [ ]:
df = load_estimated()
print(f"{len(df)} rows, {df.groupby(['InstanceType','TP','PP']).ngroups} configs")
df

## Max-Batch Summary

In [ ]:
max_batch_summary(df)

## Filter Example

In [ ]:
# Change parameters as needed
filter_df(df, instance="p5.48xlarge", tp=2, pp=4)